### Delta Live Tables - GOLD Layer

In [0]:
import dlt

In [0]:
looktable_rules ={
    "rule1":"showid is NOT NULL"
}

In [0]:
@dlt.table(
    name="gold_netflix_cast"
)
@dlt.expect_or_drop(looktable_rules)
def goldcast():
    df=spark.readStream.format("delta").load('abfss://silver@mynetflixstorage3.dfs.core.windows.net/netflix_cast')

In [0]:
@dlt.table(
    name="gold_netflix_category"
)
@dlt.expect_or_drop(looktable_rules)
def goldcategory():
    df=spark.readStream.format("delta").load('abfss://silver@mynetflixstorage3.dfs.core.windows.net/netflix_category')

In [0]:
@dlt.table(
    name="gold_netflix_countries"
)
@dlt.expect_or_drop(looktable_rules) 
def goldcountries():
    df=spark.readStream.format("delta").load('abfss://silver@mynetflixstorage3.dfs.core.windows.net/netflix_countries')

In [0]:
@dlt.table(
    name="gold_netflix_directors"
)
@dlt.expect_or_drop(looktable_rules) 
def golddirectors():
    df=spark.readStream.format("delta").load('abfss://silver@mynetflixstorage3.dfs.core.windows.net/netflix_directors')

### Staging table

In [0]:
@dlt.table

def gold_stg_netflixtitles:
    df=spark.readStream.format("delta").load('abfss://silver@mynetflixstorage3.dfs.core.windows.net/netflix_titles')
    return df

### transform View

In [0]:
from pyspark.sql.functions import *

In [0]:
@dlt.view

def gold_trns_netflixtitles:
    df=spark.readStream.table("LIVE.gold_stg_netflixtitles")
    df= df.withColumn("newflag", lit(1))
    return df

In [0]:
masterdata_rules = {
    "rule1":"newflag is NOT NULL",
    "rule2":"showid is NOT NULL"
}

In [0]:
@dlt.table

@dlt.expect_all_or_drop(masterdata_rules)
def gold_netflix_titles:
    df = spark.readStream.table("LIVE.gold_trns_netflixtitles")
    return df